In [0]:
%run /Workspace/Repos/Project_1705/Azure_Retail_Project/Project_Files/Functions/ACCESS_ADLS_GEN2_USING_SERVICE_PRINCIPALE

In [0]:
from datetime import datetime
currentdate = datetime.now().strftime("%Y%m%d")
print(currentdate)

In [0]:
# currentdate = '20260525'

In [0]:
dbutils.widgets.text("read_container_name","silver","Read Container")
read_container_name = dbutils.widgets.get("read_container_name")

In [0]:
dbutils.widgets.text("write_container_name","gold","Write Container")
write_container_name = dbutils.widgets.get("write_container_name")

In [0]:
dbutils.widgets.dropdown("table_name","category_translations",["category_translations"],"Table Name")
table_name = dbutils.widgets.get("table_name")

In [0]:

# Business Key Config
table_config = {

    "category_translations": {
        "business_key": "product_category_name"
    }
}

In [0]:
read_base_path = f"abfss://{read_container_name}@retailstorage1881.dfs.core.windows.net"

In [0]:
write_base_path = f"abfss://{write_container_name}@retailstorage1881.dfs.core.windows.net"

In [0]:
from delta.tables import DeltaTable


business_key = table_config[table_name]["business_key"]

# Paths
source_path = f"{read_base_path}/{table_name}/{currentdate}/"

target_path = f"{write_base_path}/{table_name}/"

# Read Source Data
silver_df = spark.read.format("delta").load(source_path)
silver_df.display()

try:
    dbutils.fs.ls(target_path + "/_delta_log")
    table_exists = True

except:
    table_exists = False

# First Run
if table_exists == False:

    silver_df.write \
        .format("delta") \
        .mode("overwrite") \
        .save(target_path)

    print("Initial Load Completed")

# Future Runs
else:

    print("SCD0 Incremental Load Started")

    silver_df.createOrReplaceTempView("SOURCE")

    # All Columns
    all_columns = silver_df.columns

    # Insert Columns
    insert_cols = ", ".join(all_columns)

    # Source Columns
    source_cols = ", ".join(
        [
            f"SOURCE.{col}"
            for col in all_columns
        ]
    )

    # Merge Query
    merge_query = f"""

    MERGE INTO delta.`{target_path}` TARGET

    USING SOURCE

    ON TARGET.{business_key} = SOURCE.{business_key}

    WHEN NOT MATCHED THEN

    INSERT ({insert_cols})

    VALUES ({source_cols})

    """

    spark.sql(merge_query)

    print("SCD0 Incremental Load Completed")

In [0]:
gold_df = spark.read.format("delta").load(target_path)
gold_df.display()

In [0]:
print(gold_df.count())